# 동반구매 엣지 비교 실험

| 실험 | 설명 |
|---|---|
| **exp01** Baseline | product-product 엣지 없음 |
| **exp06** Lift × α_r | co_offline + co_quick, Lift값 가중치 적용 |
| **exp07** Binary × α_r | co_offline + co_quick, 엣지 존재만 반영 (Lift 미사용) |

> 셀 순서대로 실행하면 세 실험을 순차 학습 후 최종 비교표 출력.

## 0. 환경 설정 (공통)

In [ ]:
import os, sys
import matplotlib.pyplot as plt
import matplotlib

ROOT = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

from experiments.exp_utils import (
    run_experiment, compare_experiments,
    plot_alpha_heatmap, plot_training_curve, print_metrics_table, print_recommendations
)

# 실험별 config 경로
CFG = {
    'exp01': 'experiments/configs/exp01_baseline.yaml',
    'exp06': 'experiments/configs/exp06_both_copurchase.yaml',
    'exp07': 'experiments/configs/exp07_copurchase_binary.yaml',
}

# 공통 플롯 저장 함수
def save_fig(fig, exp_name, filename):
    path = f'experiments/results/{exp_name}/{filename}'
    os.makedirs(os.path.dirname(path), exist_ok=True)
    fig.savefig(path, dpi=120, bbox_inches='tight')
    plt.show()

print('ROOT:', ROOT)

---
## 1. exp01 — Baseline (product-product 엣지 없음)

In [ ]:
r01 = run_experiment(CFG['exp01'], 'exp01_baseline')
# 재학습 강제: r01 = run_experiment(CFG['exp01'], 'exp01_baseline', force=True)

In [ ]:
print_metrics_table(r01)

In [ ]:
fig01_curve, ax = plt.subplots(figsize=(9, 4))
plot_training_curve(r01.get('history', []), ax=ax)
ax.set_title('exp01 — Baseline 학습 곡선')
save_fig(fig01_curve, 'exp01_baseline', 'training_curve.png')

fig01_alpha, ax = plt.subplots(figsize=(12, 3))
plot_alpha_heatmap('exp01_baseline', ax=ax)
save_fig(fig01_alpha, 'exp01_baseline', 'alpha_heatmap.png')

---
## 2. exp06 — Lift × α_r (co_offline + co_quick, Lift 가중치 적용)

In [ ]:
r06 = run_experiment(CFG['exp06'], 'exp06_both_copurchase')
# 재학습 강제: r06 = run_experiment(CFG['exp06'], 'exp06_both_copurchase', force=True)

In [ ]:
print_metrics_table(r06)

In [ ]:
fig06_curve, ax = plt.subplots(figsize=(9, 4))
plot_training_curve(r06.get('history', []), ax=ax)
ax.set_title('exp06 — Lift×α_r 학습 곡선')
save_fig(fig06_curve, 'exp06_both_copurchase', 'training_curve.png')

fig06_alpha, ax = plt.subplots(figsize=(12, 3))
plot_alpha_heatmap('exp06_both_copurchase', ax=ax)
ax.set_title('exp06 — α_r (co_offline vs co_quick 비교)')
save_fig(fig06_alpha, 'exp06_both_copurchase', 'alpha_heatmap.png')

---
## 3. exp07 — Binary × α_r (co_offline + co_quick, Lift 미사용)

In [ ]:
r07 = run_experiment(CFG['exp07'], 'exp07_copurchase_binary')
# 재학습 강제: r07 = run_experiment(CFG['exp07'], 'exp07_copurchase_binary', force=True)

In [ ]:
print_metrics_table(r07)

In [ ]:
fig07_curve, ax = plt.subplots(figsize=(9, 4))
plot_training_curve(r07.get('history', []), ax=ax)
ax.set_title('exp07 — Binary×α_r 학습 곡선')
save_fig(fig07_curve, 'exp07_copurchase_binary', 'training_curve.png')

fig07_alpha, ax = plt.subplots(figsize=(12, 3))
plot_alpha_heatmap('exp07_copurchase_binary', ax=ax)
ax.set_title('exp07 — α_r (Lift 미사용 시 채널 중요도)')
save_fig(fig07_alpha, 'exp07_copurchase_binary', 'alpha_heatmap.png')

---
## 4. 최종 비교

In [ ]:
df = compare_experiments(['exp01_baseline', 'exp06_both_copurchase', 'exp07_copurchase_binary'])
display(df[['exp', 'val_pr_auc', 'val_auc_roc', 'test_pr_auc', 'test_auc_roc', 'test_f1']])

In [ ]:
# exp06 vs exp07 α_r 패턴 나란히 비교
fig, axes = plt.subplots(2, 1, figsize=(14, 6))
plot_alpha_heatmap('exp06_both_copurchase', ax=axes[0])
axes[0].set_title('exp06 — Lift × α_r')
plot_alpha_heatmap('exp07_copurchase_binary', ax=axes[1])
axes[1].set_title('exp07 — Binary × α_r')
plt.tight_layout()
plt.savefig('experiments/results/alpha_compare_exp06_exp07.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# 추천 결과 비교
print('=== exp01 (Baseline) ===')
print_recommendations(r01)
print('\n=== exp06 (Lift × α_r) ===')
print_recommendations(r06)
print('\n=== exp07 (Binary × α_r) ===')
print_recommendations(r07)